In [1]:
# 1. Installe la bonne version de protobuf
!pip install "protobuf==3.20.*" --upgrade

# 2. Installe ou mets à jour les librairies critiques
!pip install -q transformers datasets accelerate seqeval evaluate --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
onnx 1.20.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
ray 2.52.1 requires click!=8.3.*,>=7.0, but you have click 8.3.1 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.

In [2]:
!pip install -U accelerate transformers

Chargement et Restructuration des Données

In [10]:
import numpy as np
from datasets import load_dataset, Dataset

# 1. Charger le dataset
raw_dataset = load_dataset("asas-ai/ANERCorp")

# 2. Fonction optimisée pour grouper les mots en phrases
# On coupe si on rencontre un point "." ou si la phrase dépasse 64 mots
def reform_sentences(dataset):
    sentences = []
    labels = []
    
    current_sent = []
    current_labels = []
    
    for word, tag in zip(dataset['word'], dataset['tag']):
        current_sent.append(word)
        current_labels.append(tag)
        
        # Condition de fin de phrase : Point ou longueur > 64
        if word == "." or len(current_sent) >= 64:
            sentences.append(current_sent)
            labels.append(current_labels)
            current_sent = []
            current_labels = []
            
    # Ajouter le reste s'il y en a
    if current_sent:
        sentences.append(current_sent)
        labels.append(current_labels)
        
    return {"tokens": sentences, "ner_tags": labels}

# Appliquer la transformation
print("Restructuration des données en cours...")
train_data = reform_sentences(raw_dataset['train'])
test_data = reform_sentences(raw_dataset['test'])

# Re-créer les objets Dataset HuggingFace
train_dataset = Dataset.from_dict(train_data)
test_dataset = Dataset.from_dict(test_data)

print(f"Nombre de phrases d'entraînement : {len(train_dataset)}")
print(f"Exemple : {train_dataset[0]}")

Restructuration des données en cours...
Nombre de phrases d'entraînement : 4440
Exemple : {'tokens': ['فرانكفورت', '(د', 'ب', 'أ)', 'أعلن', 'اتحاد', 'صناعة', 'السيارات', 'في', 'ألمانيا', 'امس', 'الاول', 'أن', 'شركات', 'صناعة', 'السيارات', 'في', 'ألمانيا', 'تواجه', 'عاما', 'صعبا', 'في', 'ظل', 'ركود', 'السوق', 'الداخلية', 'والصادرات', 'وهي', 'تسعي', 'لان', 'يبلغ', 'الانتاج', 'حوالي', 'خمسة', 'ملايين', 'سيارة', 'في', 'عام', '2002', '.'], 'ner_tags': ['B-LOC', 'O', 'O', 'O', 'O', 'B-ORG', 'I-ORG', 'I-ORG', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}


Tokenization et Alignement

In [11]:
from transformers import AutoTokenizer

# Utilisation d'AraBERT (bien meilleur que mBERT pour l'arabe)
model_checkpoint = "aubmindlab/bert-base-arabertv02" 
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Création des dictionnaires de labels
label_list = sorted(list(set([tag for tags in train_dataset["ner_tags"] for tag in tags])))
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(f"Labels : {label_list}")

# Fonction d'alignement des labels
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], 
        truncation=True, 
        is_split_into_words=True,
        max_length=128, # On limite la taille pour la mémoire
        padding="max_length" 
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Ignorer les tokens spéciaux [CLS], [SEP]
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]]) # Premier sous-mot du mot
            else:
                label_ids.append(-100) # Ignorer les sous-mots suivants
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Appliquer la tokenization
tokenized_train = train_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_test = test_dataset.map(tokenize_and_align_labels, batched=True)

Labels : ['B-LOC', 'B-MISC', 'B-ORG', 'B-PERS', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PERS', 'O']


Map:   0%|          | 0/4440 [00:00<?, ? examples/s]

Map:   0%|          | 0/988 [00:00<?, ? examples/s]

Configuration des Métriques

In [12]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Convertir les IDs en vrais labels, en ignorant les -100
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

Entraînement du Modèle

In [13]:
#### from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint, 
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# Arguments d'entraînement
args = TrainingArguments(
    output_dir="ner-arabert-finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none",
    
    # --- AJOUTS IMPORTANTS ---
    load_best_model_at_end=True,     # Charge le meilleur modèle à la fin
    metric_for_best_model="eval_loss", # Le critère pour décider du "meilleur" (ici la perte minimale)
    greater_is_better=False,         # Pour la Loss, on veut qu'elle soit la plus petite possible
    save_total_limit=2               # (Optionnel) Ne garde que les 2 meilleurs checkpoints pour économiser de la place
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Lancer l'entraînement
print("Démarrage de l'entraînement...")
trainer.train()

Some weights of BertForTokenClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_55/3474972961.py:32: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Démarrage de l'entraînement...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.084000,0.143535,0.840525,0.783559,0.811043,0.965251
2,0.049900,0.140075,0.832729,0.803236,0.817716,0.968450
3,0.034600,0.142497,0.836947,0.810232,0.823373,0.969530


TrainOutput(global_step=834, training_loss=0.08038023550161641, metrics={'train_runtime': 180.163, 'train_samples_per_second': 73.933, 'train_steps_per_second': 4.629, 'total_flos': 870173266728960.0, 'train_loss': 0.08038023550161641, 'epoch': 3.0})

Test

In [15]:
from transformers import pipeline

# 1. Sauvegarder le modèle
trainer.save_model("./mon_modele_ner_arabe")
tokenizer.save_pretrained("./mon_modele_ner_arabe")

# 2. Tester avec une pipeline
nlp = pipeline("ner", model="./mon_modele_ner_arabe", tokenizer="./mon_modele_ner_arabe", aggregation_strategy="simple")

# Note: AraBERT fonctionne mieux sur l'arabe, essaie un texte arabe :
text_long = """
أعلن المتحدث باسم الأمم المتحدة ستيفان دوجاريك اليوم في نيويورك أن الأمين العام أنطونيو غوتيريش سيتوجه غداً إلى العاصمة اللبنانية بيروت. 
ومن المقرر أن يلتقي غوتيريش بالرئيس ميشال عون ورئيس البرلمان نبيه بري لبحث سبل دعم المنظمة الدولية للشعب اللبناني في ظل الأزمة الاقتصادية الراهنة، 
كما سيزور مقر قوات اليونيفيل في الجنوب لتفقد الأوضاع هناك.
"""
results = nlp(text_long)
print(results)

Device set to use cuda:0


[{'entity_group': 'ORG', 'score': np.float32(0.9898511), 'word': 'الأمم المتحدة', 'start': 19, 'end': 32}, {'entity_group': 'PERS', 'score': np.float32(0.98644656), 'word': 'ستيفان دوجاريك', 'start': 33, 'end': 47}, {'entity_group': 'LOC', 'score': np.float32(0.99409807), 'word': 'نيويورك', 'start': 57, 'end': 64}, {'entity_group': 'PERS', 'score': np.float32(0.97638184), 'word': 'أنطونيو غوتيريش', 'start': 81, 'end': 96}, {'entity_group': 'LOC', 'score': np.float32(0.9927779), 'word': 'بيروت', 'start': 131, 'end': 136}, {'entity_group': 'PERS', 'score': np.float32(0.9420488), 'word': 'غوتيريش', 'start': 159, 'end': 166}, {'entity_group': 'PERS', 'score': np.float32(0.98312443), 'word': 'ميشال عون', 'start': 175, 'end': 184}, {'entity_group': 'PERS', 'score': np.float32(0.9814103), 'word': 'نبيه بري', 'start': 200, 'end': 208}, {'entity_group': 'ORG', 'score': np.float32(0.5911996), 'word': 'المنظمة', 'start': 222, 'end': 229}, {'entity_group': 'ORG', 'score': np.float32(0.5481409), 'w